# 9.2 融化层自动检测

**参考文献**：Ryzhkov & Zrnic, Chapter 9, pp. 901-930

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def detect_melting_layer(Z_profile, ZDR_profile, rho_profile):
    """
    融化层检测算法
    基于Z峰值、ZDR梯度和ρhv谷值
    """
    n = len(Z_profile)
    
    # Z的二阶导数找峰值
    Z_gradient = np.gradient(Z_profile)
    Z_curvature = np.gradient(Z_gradient)
    
    # 融化层指标
    melting_indicator = np.zeros(n)
    
    # Z峰值
    z_peak = Z_gradient < 0  # 上升后下降
    
    # ρhv谷值
    rho_low = rho_profile < 0.92
    
    # 综合指标
    melting_indicator = z_peak.astype(float) * 0.5 + rho_low.astype(float) * 0.5
    
    return melting_indicator

def plot_melting_detection():
    """融化层检测示意"""
    height = np.linspace(0.5, 5, 100)
    
    # 生成典型廓线
    Z = np.zeros_like(height)
    ZDR = np.zeros_like(height)
    rho = np.ones_like(height) * 0.97
    
    # 设定融化层
    melt_base = 2.0
    melt_top = 3.5
    
    for i, h in enumerate(height):
        if h < melt_base:
            Z[i] = 25 + h * 4
            ZDR[i] = 0.5
        elif h < melt_top:
            frac = (h - melt_base) / (melt_top - melt_base)
            Z[i] = 40 + 10 * np.sin(frac * np.pi)
            ZDR[i] = 2.5 + 1.5 * frac
            rho[i] = 0.88 + 0.1 * frac
        else:
            Z[i] = 35 + (h - melt_top) * 2
            ZDR[i] = 3.5 + (h - melt_top) * 0.3
            rho[i] = 0.97
    
    melting_ind = detect_melting_layer(Z, ZDR, rho)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 6))
    
    ax1 = axes[0]
    ax1.plot(Z, height, 'b-', linewidth=2)
    ax1.axhline(y=melt_base, color='green', linestyle='--')
    ax1.axhline(y=melt_top, color='red', linestyle='--')
    ax1.set_xlabel('Z (dBZ)', fontsize=11)
    ax1.set_ylabel('高度 (km)', fontsize=11)
    ax1.set_title('反射率廓线', fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[1]
    ax2.plot(ZDR, height, 'g-', linewidth=2)
    ax2.axhline(y=melt_base, color='green', linestyle='--')
    ax2.axhline(y=melt_top, color='red', linestyle='--')
    ax2.set_xlabel('ZDR (dB)', fontsize=11)
    ax2.set_title('差分反射率廓线', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    ax3 = axes[2]
    ax3.plot(rho, height, 'r-', linewidth=2)
    ax3.axhline(y=melt_base, color='green', linestyle='--')
    ax3.axhline(y=melt_top, color='red', linestyle='--')
    ax3.set_xlabel('|ρhv|', fontsize=11)
    ax3.set_title('相关系数廓线', fontsize=12)
    ax3.grid(True, alpha=0.3)
    
    ax4 = axes[3]
    ax4.plot(melting_ind, height, 'k-', linewidth=2)
    ax4.fill_betweenx(height, melting_ind, alpha=0.3)
    ax4.axhline(y=melt_base, color='green', linestyle='--')
    ax4.axhline(y=melt_top, color='red', linestyle='--')
    ax4.set_xlabel('融化层指标', fontsize=11)
    ax4.set_title('融化层检测', fontsize=12)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n检测到融化层: {melt_base:.1f} - {melt_top:.1f} km")

In [ ]:
plot_melting_detection()

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 9, Springer.